# 02 — Embedding Pipeline

Build and validate the FAISS vector index:
1. Load raw API data from `data/raw/`
2. Run preprocessor to create unified dataset
3. Embed with `all-MiniLM-L6-v2`
4. Build and persist FAISS index
5. Validate retrieval with sample queries

> **Prerequisite:** Run `01_api_exploration.ipynb` (or `api_fetcher.py`) first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded.')

## 1. Preprocess raw data

In [ ]:
from backend.data.preprocessor import build_unified_dataset

records = build_unified_dataset()
print(f'Unified dataset: {len(records)} records')
print('\nSample record keys:', list(records[0].keys()) if records else 'empty')

In [ ]:
# Inspect first record
if records:
    import json
    print(json.dumps(records[0], indent=2))

## 2. Check overview coverage

In [ ]:
has_overview = sum(1 for r in records if len(r.get('overview', '')) > 20)
has_genres   = sum(1 for r in records if r.get('genres'))
has_platform = sum(1 for r in records if r.get('platforms'))
print(f'Records with overview : {has_overview}/{len(records)}')
print(f'Records with genres   : {has_genres}/{len(records)}')
print(f'Records with platforms: {has_platform}/{len(records)}')

## 3. Build FAISS index

In [ ]:
from backend.data.embeddings.embed_builder import build_faiss_index

# This will take a few minutes depending on dataset size
build_faiss_index(records=records)
print('FAISS index built and saved.')

## 4. Validate embedding shape

In [ ]:
from backend.rag.embed_query import embed_query, embed_queries
import numpy as np

vec = embed_query('emotional healing drama')
print(f'Single query embedding shape : {vec.shape}')   # (1, 384)
print(f'dtype                        : {vec.dtype}')   # float32
print(f'L2 norm                      : {np.linalg.norm(vec):.4f}')  # ~1.0

vecs = embed_queries(['happy films', 'dark thriller', 'comedy series'])
print(f'Batch embedding shape        : {vecs.shape}')  # (3, 384)

## 5. Validate FAISS retrieval

In [ ]:
from backend.rag.faiss_retriever import FAISSRetriever

retriever = FAISSRetriever(top_k=5)
results = retriever.search(['melancholic romance', 'bittersweet love story'])

print(f'Results returned: {len(results)}')
for r in results:
    print(f"  • {r['title']} ({r.get('year','?')}) — score: {r.get('_retrieval_score')}")

In [ ]:
# Test exclusion filter
excluded = [results[0]['title']] if results else []
results2 = retriever.search(['sad quiet film'], excluded_titles=excluded)
returned_titles = [r['title'] for r in results2]
assert excluded[0] not in returned_titles if excluded else True
print(f'Exclusion filter works ✓  —  excluded: {excluded}')

## 6. Summary

- Unified dataset built ✓
- FAISS index persisted ✓
- Embedding shape validated ✓
- Retrieval working ✓

Proceed to `03_end_to_end_demo.ipynb` for a full system walkthrough.